# SPRKD Quickstart Notebook

Modern, package-based entry point for SPRKD. Mirrors the end-to-end pipeline of the original `SPRKD.ipynb` but uses the public `sprkd` package APIs and runs unchanged on CPU / CUDA / Apple Silicon (MPS).

For the unmodified ISEF 2023 implementation see [`SPRKD.ipynb`](./SPRKD.ipynb).

## Pipeline

1. Configure the malaria DataLoader (capped to a few batches for fast execution).
2. Train a weak teacher with saddle tracking.
3. Aggregate the recorded saddles into an Approximated Saddle Region (ASR).
4. Inject the ASR into the 4x-smaller student via `inject_state_list` (TLI).
5. Run the SPRKD student loop (Transformation Matrix -> NHE -> PGD).
6. Compare against the released checkpoints on `TESTSET.pth`.

**Note**: this notebook uses a tiny number of steps (just enough to demonstrate every code path) so it completes in ~1 minute. For the full paper reproduction use `scripts/reproduce_malaria.py`.

In [ ]:
import torch
import torch.nn as nn

from sprkd import (
    MalariaTeacherCNN, MalariaStudentCNN,
    aggregate_asr, set_seed,
    load_legacy_student,
)
from sprkd.tli import inject_state_list
from sprkd.data import MalariaDataConfig, find_default_root, make_dataloaders
from sprkd.training import train_teacher, train_student
from sprkd.utils import get_device

set_seed(0)
device = get_device()
print('device:', device)

## 1. DataLoader (capped to N batches per epoch)

In [ ]:
cfg = MalariaDataConfig(
    root=find_default_root(),
    batch_size=64,
    num_workers=0,
    seed=0,
)
train_loader, valid_loader, full = make_dataloaders(cfg)
print('total cells:', len(full))

class _Capped:
    """DataLoader wrapper that truncates each epoch to N batches."""
    def __init__(self, loader, n):
        self._loader, self._n = loader, n
    def __iter__(self):
        for i, batch in enumerate(self._loader):
            if i >= self._n:
                break
            yield batch
    def __len__(self):
        return min(self._n, len(self._loader))

fast_train = _Capped(train_loader, 8)
fast_valid = _Capped(valid_loader, 4)

## 2. Train the (weak) teacher with saddle tracking

`saddle_steps=4` checks for strong saddles every four mini-batches; the strong-saddle criterion follows paper Equation 1.

In [ ]:
teacher = MalariaTeacherCNN().to(device)
sprkd_t, t_history = train_teacher(
    teacher, fast_train, fast_valid,
    loss_fn=nn.CrossEntropyLoss(),
    n_epochs=1,
    saddle_steps=4,
    device=device,
    progress=False,
)
print(f'recorded {len(sprkd_t.saddle_repository)} saddles')
print(f'best teacher val acc: {t_history.best_valid_acc():.2f}%')

## 3. Build the ASR + inject into the student

In [ ]:
if len(sprkd_t.saddle_repository) == 0:
    sprkd_t.saddle_repository.append(list(teacher.parameters()), loss=0.0)
asr = aggregate_asr([sprkd_t.saddle_repository.snapshots])
print('ASR layers:', len(asr))

student = MalariaStudentCNN().to(device)
inject_state_list(student, asr, teacher=teacher)
targets = [p.detach().clone() for p in student.parameters()]
print('student params:', sum(p.numel() for p in student.parameters()))

## 4. SPRKD student training (Transformation Matrix -> NHE -> PGD)

In [ ]:
sprkd_s, s_history = train_student(
    student, fast_train, fast_valid,
    loss_fn=nn.CrossEntropyLoss(),
    teacher_saddle_points=targets,
    n_epochs=1,
    device=device,
    progress=False,
    sprkd_kwargs={'max_nhe_steps': 0},
)
print(f'best SPRKD student val acc: {s_history.best_valid_acc():.2f}%')

## 5. Compare against the released ISEF 2023 checkpoints

The `MODELS/` directory contains the original released checkpoints. Loading them through `sprkd.load_legacy_student` extracts the underlying `nn.Module` from the legacy fastai `Learner` pickles, side-stepping the `__main__.SPRKD` import error.

In [ ]:
from pathlib import Path

MODELS = {
    'SPRKD':   Path('..') / 'MODELS' / 'SPRKD_MALARIA.pth',
    'Control': Path('..') / 'MODELS' / 'CONTROL_MALARIA.pth',
    'RKD':     Path('..') / 'MODELS' / 'RKD_MALARIA_STUDENT.pth',
}

ts = torch.load(Path('..') / 'TESTSET.pth', map_location='cpu', weights_only=False)
xs, ys = ts[0].to(device), ts[1].to(device)
print('held-out test set size:', xs.shape[0])

for label, path in MODELS.items():
    if not path.is_file():
        print(f'  {label}: MISSING ({path})')
        continue
    m = load_legacy_student(path).to(device).eval()
    with torch.no_grad():
        preds = torch.argmax(m(xs), dim=1)
    acc = 100.0 * (preds == ys).float().mean().item()
    print(f'  {label}: {acc:.2f}%')